In [ ]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb")

In [ ]:
dataset["train"][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
def preprocess_function(examples):
  return tokenizer(
      examples["text"],
      truncation = True
  )

tokenized_dataset = dataset.map(
    preprocess_function,
    batched = True
)

In [ ]:
split = dataset["train"].train_test_split(
    test_size = 0.1,
    seed = 42
)

train_dataset = split["train"]
validation_dataset = split["test"]

test_dataset = dataset["test"]

In [ ]:
train_dataset = train_dataset.map(
    preprocess_function,
    batched = True
)

validation_dataset = validation_dataset.map(
    preprocess_function,
    batched = True
)

test_dataset = test_dataset.map(
    preprocess_function,
    batched = True
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [ ]:
id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels = 2,
    id2label = id2label,
    label2id = label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./imdb_sentiment_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis = -1)

  accuracy_score = accuracy.compute(predictions = predictions, references = labels)["accuracy"]
  f1_score = f1.compute(predictions = predictions, references = labels)["f1"]

  return {
      "accuracy": accuracy_score,
      "f1": f1_score
  }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.232102,0.229494,0.908800,0.905079
2,0.153665,0.263836,0.920800,0.920800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2814, training_loss=0.2075815573599462, metrics={'train_runtime': 2451.9119, 'train_samples_per_second': 18.353, 'train_steps_per_second': 1.148, 'total_flos': 5896822472726160.0, 'train_loss': 0.2075815573599462, 'epoch': 2.0})

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.153665,0.199152,2,0.924080,0.922015


{'eval_loss': 0.19915175437927246, 'eval_accuracy': 0.92408, 'eval_f1': 0.9220149560358287}


In [ ]:
trainer.save_model("./imdb_sentiment_model")
tokenizer.save_pretrained("./imdb_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./imdb_sentiment_model/tokenizer_config.json',
 './imdb_sentiment_model/tokenizer.json')

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

model_path = "./imdb_sentiment_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

classifier = pipeline(
    "text-classification",
    model = model,
    tokenizer = tokenizer,
)

print(classifier("This movie was absolutely fantastic!"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9876701831817627}]


In [ ]:
from datasets import load_dataset
from evaluate import evaluator
import evaluate

# Randomly select 1,000 reviews from the test set
test_data = load_dataset(
    "stanfordnlp/imdb",
    split="test"
).shuffle(seed=42).select(range(1000))

# Check that both sentiment labels are present
print("Label counts:", {
    "negative": test_data["label"].count(0),
    "positive": test_data["label"].count(1)
})

task_evaluator = evaluator("text-classification")

metrics = evaluate.combine([
    "accuracy",
    "precision",
    "recall",
    "f1"
])

results = task_evaluator.compute(
    model_or_pipeline=classifier,
    data=test_data,
    metric=metrics,
    label_mapping={
        "NEGATIVE": 0,
        "POSITIVE": 1
    }
)

print(results)

Label counts: {'negative': 512, 'positive': 488}
{'accuracy': 0.914, 'precision': 0.9294871794871795, 'recall': 0.8913934426229508, 'f1': 0.9100418410041841, 'total_time_in_seconds': 12.902382000000216, 'samples_per_second': 77.5050684439496, 'latency_in_seconds': 0.012902382000000216}


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import json

# Create a permanent project folder in Google Drive
project_dir = "/content/drive/MyDrive/IMDb_Sentiment_Analysis"
model_dir = os.path.join(project_dir, "imdb_sentiment_model")

os.makedirs(model_dir, exist_ok=True)

# Save the fine-tuned model and tokenizer
classifier.model.save_pretrained(model_dir)
classifier.tokenizer.save_pretrained(model_dir)

# Save the evaluation metrics from our previous cell
with open(os.path.join(project_dir, "evaluation_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("Model and evaluation results saved!")
print("Project folder:", project_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and evaluation results saved!
Project folder: /content/drive/MyDrive/IMDb_Sentiment_Analysis


In [ ]:
readme = """
# IMDb Sentiment Analysis using Hugging Face

## Project Overview
This project classifies movie reviews as positive or negative using
a fine-tuned DistilBERT model from Hugging Face Transformers.

## Technologies Used
- Python
- Google Colab
- Hugging Face Transformers
- Hugging Face Datasets
- Evaluate
- DistilBERT

## Workflow
1. Load the IMDb movie-review dataset.
2. Tokenize movie reviews using a pretrained tokenizer.
3. Fine-tune DistilBERT for binary sentiment classification.
4. Save the trained model and tokenizer.
5. Predict sentiment for new movie reviews.
6. Evaluate the model on a randomly shuffled sample
   of 1,000 held-out test reviews.

## Evaluation Results
- Accuracy: 91.4%
- Precision: 92.95%
- Recall: 89.14%
- F1 score: 91.00%

These results are from a 1,000-review test sample.
They should not be interpreted as results on the full test set.

## Example Inference
Input:
"This movie was absolutely fantastic!"

Expected type of output:
A sentiment label (POSITIVE or NEGATIVE) and a confidence score.

## Project Artifacts
- imdb_sentiment_model/: saved model and tokenizer
- evaluation_results.json: evaluation metrics
- README.md: project documentation

## How to Use
Load the saved model and tokenizer using
AutoModelForSequenceClassification and AutoTokenizer,
then create a text-classification pipeline.

## Limitations
- Evaluation was performed on a sample, not the entire test set.
- Predictions can be incorrect, even when confidence is high.
- Performance may vary on reviews unlike those in the IMDb dataset.
"""

readme_path = os.path.join(project_dir, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme.strip() + "\n")

print("README saved to:", readme_path)


README saved to: /content/drive/MyDrive/IMDb_Sentiment_Analysis/README.md


In [ ]:
for root, dirs, files in os.walk(project_dir):
    level = root.replace(project_dir, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")

    if level >= 1:
        dirs[:] = []

    for filename in files:
        print("  " * (level + 1) + filename)


IMDb_Sentiment_Analysis/
  evaluation_results.json
  README.md
  imdb_sentiment_model/
    config.json
    model.safetensors
    tokenizer_config.json
    tokenizer.json


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

model_dir = "/content/drive/MyDrive/IMDb_Sentiment_Analysis/imdb_sentiment_model"

saved_tokenizer = AutoTokenizer.from_pretrained(model_dir)
saved_model = AutoModelForSequenceClassification.from_pretrained(model_dir)

saved_classifier = pipeline(
    "text-classification",
    model=saved_model,
    tokenizer=saved_tokenizer
)

print("Saved model loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Saved model loaded successfully!


In [ ]:

reviews = [
    "This movie was brilliant, emotional, and beautifully acted.",
    "The plot was boring, the acting was terrible, and I hated it."
]

predictions = saved_classifier(reviews)

for review, prediction in zip(reviews, predictions):
    print("Review:", review)
    print("Predicted sentiment:", prediction["label"])
    print("Score:", round(prediction["score"], 4))
    print("-" * 50)


Review: This movie was brilliant, emotional, and beautifully acted.
Predicted sentiment: POSITIVE
Score: 0.9934
--------------------------------------------------
Review: The plot was boring, the acting was terrible, and I hated it.
Predicted sentiment: NEGATIVE
Score: 0.9933
--------------------------------------------------
